# Audiolivro com leitura guiada: Memórias do Subsolo

Fiódor Dostoiévski (1864, domínio público). Fonte: Записки из подполья, Часть I, Викитека (ru.wikisource.org), texto da revista Эпоха, 1864. Tradução nossa do russo, conferida com a tradução inglesa de Constance Garnett (1918, domínio público).

Duas vozes, nunca trocadas: o **leitor** lê só o texto do autor, e o **comentarista** entra entre os trechos com contexto, ligações dentro da obra e com outras obras. Cada corte é anunciado pelo leitor entre parênteses.

**Antes de começar:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Depois rode as células de cima para baixo. Rodar de novo qualquer célula é seguro.

**Saída** (no seu Google Drive, em `MyDrive/Audiolivros-pessoais/subsolo/capNN/`), para cada capítulo:
- `capNN_guiada.mp3`: texto e comentários, com um sinal curto antes e depois de cada comentário;
- `capNN_limpa.mp3`: só o texto do autor, para reouvir sem comentários.

**Cortes deste lote:**
- Parte I, cap. I: os gritos contra os velhos respeitáveis (1 frase).
- Parte I, cap. I: a criada e os conselhos para sair de Petersburgo (2 frases).
- Parte I, cap. II: as leis da consciência intensificada e o consolo do canalha (2 frases).

**Licença:** a voz padrão (OmniVoice) é de uso **não comercial** (CC-BY-NC). As vozes são sintéticas, sem clonar ninguém.

In [ ]:
# ── 1. Conferir a GPU ───────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("ATENÇÃO: sem GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente "
          "de execução -> GPU T4, e rode o notebook de novo do início.")

In [ ]:
# ── 2. Google Drive e pastas ────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount("/content/drive")

CAPITULOS_PARA_GERAR = [1, 2]   # quais capítulos narrar (ex.: [2])

LIVRO = "/content/drive/MyDrive/Audiolivros-pessoais/subsolo"
PASTA_VOZES = os.path.join(LIVRO, "vozes")
os.makedirs(PASTA_VOZES, exist_ok=True)
print("Os arquivos vão para:", LIVRO)

In [ ]:
# ── 3. Instalar o VoiceStudio (primeira vez: ~5-8 min) ──────────────────────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sdpaula36-stack/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    if subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"\nFALHOU: {what or shown}. Veja o erro acima; rodar de novo é seguro.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"], what="git pull")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="instalação do backend")

from huggingface_hub import snapshot_download
print("Baixando o modelo de voz (alguns GB, só na primeira vez)...")
snapshot_download("k2-fsa/OmniVoice")
print("Pronto.")

In [ ]:
# ── 4. Ligar o servidor do VoiceStudio ──────────────────────────────────────
import json, os, subprocess, sys, time, urllib.request

PORT = 3900
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = "/content/voicestudio_backend.log"

def health():
    try:
        with urllib.request.urlopen(f"{BASE}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if not info:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    env = os.environ.copy()
    env.update(OMNIVOICE_SERVER_MODE="1", PYTHONUNBUFFERED="1")
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content/VoiceStudio", env=env,
        stdout=open(LOG_PATH, "ab"), stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and not info:
        print(".", end="", flush=True)
        time.sleep(3)
        info = health()
    print()
    if not info:
        raise SystemExit("O servidor não ligou em 5 min. Final do log:\n"
                         + "".join(open(LOG_PATH, errors="replace").readlines()[-40:]))
print("Servidor ligado:", info)

In [ ]:
# ── 4.5. Aceitar as licenças dos modelos (obrigatório antes de gerar) ───────
# Sem este passo o /generate falha. Rodar esta célula aceita as licenças em seu nome:
# OmniVoice é CC-BY-NC (uso NÃO comercial: o áudio não pode ser vendido).
import requests
BASE = "http://127.0.0.1:3900"

for repo in ("k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"):
    st = requests.get(f"{BASE}/models/licenses/acceptance/{repo}", timeout=30).json()
    if st.get("accepted"):
        print(f"[ok] {repo} já aceito")
        continue
    r = requests.post(
        f"{BASE}/models/licenses/accept",
        json={"repo_id": repo, "fingerprint": st["fingerprint"], "accepted": True},
        timeout=30,
    )
    r.raise_for_status()
    print(f"[aceito] {repo} ({st.get('license')})")

## Roteiro

Cada trecho é `[voz, texto, velocidade, pausa depois em segundos]`, com voz `leitor` ou `comentarista`.

- O **leitor** só lê frases do autor. Um corte aparece como `(Corte: ...)` e fica nas duas versões.
- O **comentarista** só entra entre trechos, nunca no meio de uma frase do autor.
- **voz** aceita só estes termos: `male`/`female`; `child`, `teenager`, `young adult`, `middle-aged`, `elderly`; `very low pitch` … `very high pitch`; `whisper`. Se não gostar de uma voz, troque a `semente`.

Edite à vontade. Os trechos que você mudar são gerados de novo na célula 7.

In [ ]:
# ── 5. Roteiro (EDITE AQUI) ─────────────────────────────────────────────────
VOZES = {'leitor': {'voz': 'male, middle-aged, low pitch', 'semente': 601},
 'comentarista': {'voz': 'female, young adult, moderate pitch', 'semente': 611}}

CAPITULOS = {
    1: {"titulo": "O homem doente", "trechos": [
        ["comentarista", "Memórias do Subsolo, de Fiódor Dostoiévski, em leitura guiada. Parte um, capítulo um.", 1.0, 0.8],
        ["comentarista", "Eu sou a voz dos comentários. A outra voz, a do leitor, só lê o texto de Dostoiévski, numa tradução nossa, feita do russo. Quando houver um corte, o leitor avisa entre parênteses.", 1.0, 0.5],
        ["comentarista", "O livro saiu em 1864, na revista Época, dos irmãos Dostoiévski. Foi um ano duro para ele: a primeira mulher morreu em abril, e o irmão Mikhail, em julho.", 1.0, 0.5],
        ["comentarista", "Antes do primeiro capítulo, há uma nota do próprio autor. Ela importa, porque avisa que quem fala no livro não é Dostoiévski.", 1.0, 1.0],
        ["leitor", "Nota do autor.", 0.95, 0.6],
        ["leitor", "Tanto o autor destas memórias quanto as próprias memórias são, evidentemente, inventados. Mesmo assim, pessoas como o autor de memórias assim não só podem como até devem existir na nossa sociedade,", 0.95, 0.5],
        ["leitor", "se levarmos em conta as circunstâncias em que a nossa sociedade, de modo geral, se formou.", 0.95, 0.5],
        ["leitor", "Eu quis mostrar ao público, com mais destaque do que de costume, um dos caracteres de um tempo que acabou de passar. É um dos representantes de uma geração que ainda vive os seus últimos dias.", 0.95, 0.5],
        ["leitor", "Neste fragmento, intitulado O subsolo, essa pessoa apresenta a si mesma e ao seu modo de ver, e como que quer explicar as causas pelas quais ela surgiu, e tinha de surgir, no nosso meio.", 0.95, 0.5],
        ["leitor", "No fragmento seguinte virão as verdadeiras memórias dessa pessoa, sobre alguns acontecimentos da sua vida. Fiódor Dostoiévski.", 0.95, 1.2],
        ["leitor", "Parte um. O subsolo. Capítulo um.", 0.95, 1.0],
        ["leitor", "Sou um homem doente... Sou um homem mau. Um homem desagradável, é o que eu sou. Acho que estou doente do fígado.", 0.95, 0.6],
        ["leitor", "Aliás, não entendo patavina da minha doença, e não sei ao certo o que me dói. Não me trato e nunca me tratei, embora respeite a medicina e os médicos.", 0.95, 0.5],
        ["leitor", "Além disso, sou supersticioso ao extremo; bem, pelo menos o bastante para respeitar a medicina. (Sou instruído o bastante para não ser supersticioso, mas sou supersticioso.)", 0.95, 0.5],
        ["leitor", "Não, senhores, eu não quero me tratar por raiva. Isso, com certeza, os senhores não vão ter a bondade de entender. Pois eu entendo.", 0.95, 0.5],
        ["leitor", "Eu, naturalmente, não saberia explicar a quem exatamente vou atingir com a minha raiva. Sei perfeitamente que não vou aprontar nada contra os médicos por não me tratar com eles;", 0.95, 0.5],
        ["leitor", "sei melhor do que ninguém que, com tudo isso, só vou prejudicar a mim mesmo, e a mais ninguém. Mas, mesmo assim, se não me trato, é por raiva. O fígado dói? Pois que doa ainda mais!", 0.95, 0.8],
        ["leitor", "Vivo assim há muito tempo: uns vinte anos. Agora tenho quarenta. Antes eu trabalhava numa repartição; agora não trabalho mais. Fui um funcionário mau. Era grosseiro, e tinha prazer nisso.", 0.95, 0.5],
        ["leitor", "Afinal, eu não aceitava propina, então precisava me recompensar pelo menos com isso.", 0.95, 0.5],
        ["leitor", "(Piada ruim; mas não vou riscá-la. Escrevi achando que ia sair muito espirituosa; e agora que eu mesmo vi que só queria me exibir de um jeito baixo, de propósito não vou riscá-la!)", 0.95, 0.5],
        ["leitor", "Quando os requerentes se aproximavam da minha mesa para pedir informações, eu rangia os dentes para eles, e sentia um prazer implacável quando conseguia magoar alguém. Quase sempre conseguia.", 0.95, 0.5],
        ["leitor", "Na maioria era gente tímida: requerentes, já se sabe. Mas, entre os janotas, havia um oficial que eu não suportava. Ele não queria de jeito nenhum se submeter, e fazia um barulho repugnante com o sabre.", 0.95, 0.5],
        ["leitor", "Por causa desse sabre, ficamos um ano e meio em guerra. No fim, eu venci. Ele parou de fazer barulho. Aliás, isso foi ainda na minha juventude.", 0.95, 0.5],
        ["leitor", "Mas sabem, senhores, qual era o ponto principal da minha raiva? Pois é aí que estava toda a questão, aí é que estava a maior torpeza:", 0.95, 0.5],
        ["leitor", "a todo instante, mesmo no momento de maior bile, eu tinha a consciência vergonhosa de que não só não era um homem mau, como nem sequer era um homem amargurado; de que só espantava os pardais à toa, e com isso me divertia.", 0.95, 0.5],
        ["leitor", "Estou espumando pela boca, mas tragam-me uma bonequinha qualquer, deem-me um chazinho com açúcar, e eu talvez até me acalme. Até me comovo de coração,", 0.95, 0.5],
        ["leitor", "embora depois, com certeza, vá ranger os dentes contra mim mesmo, e passe alguns meses sem dormir, de vergonha. Esse é o meu jeito.", 0.95, 0.8],
        ["leitor", "Isso que eu disse há pouco, que fui um funcionário mau, foi mentira. Menti por raiva. Eu só me divertia, com os requerentes e com o oficial, mas no fundo nunca consegui me tornar mau.", 0.95, 0.5],
        ["leitor", "A todo instante eu sentia em mim muitos e muitos elementos contrários a isso. Sentia que eles fervilhavam dentro de mim, esses elementos contrários.", 0.95, 0.5],
        ["leitor", "Sabia que fervilharam em mim a vida inteira, e que pediam para sair, mas eu não deixava, não deixava, de propósito não deixava que saíssem.", 0.95, 0.5],
        ["leitor", "Eles me torturavam até a vergonha; me levavam a convulsões e, por fim, me cansaram. E como me cansaram!", 0.95, 0.5],
        ["leitor", "Será que não lhes parece, senhores, que agora estou me arrependendo de alguma coisa diante dos senhores, que estou pedindo perdão por alguma coisa?... Tenho certeza de que lhes parece...", 0.95, 0.5],
        ["leitor", "Mas, aliás, garanto que para mim tanto faz, se lhes parece...", 0.95, 1.2],
        ["comentarista", "Repare no método do narrador. Ele abre dizendo: sou um homem mau. Dois parágrafos depois: era mentira, menti por raiva.", 1.0, 0.5],
        ["comentarista", "Ele afirma, desmente e ainda adivinha o que você está pensando dele. Esse vai e vem é o livro inteiro.", 1.0, 0.5],
        ["comentarista", "Uma leitura possível: ele não consegue falar sem imaginar alguém que o julga. Por isso nunca diz nada de uma vez só.", 1.0, 1.2],
        ["leitor", "Eu não só não consegui me tornar mau, como não consegui me tornar coisa nenhuma: nem mau, nem bom, nem canalha, nem honesto, nem herói, nem inseto.", 0.95, 0.5],
        ["leitor", "Agora vou levando a vida no meu canto, provocando a mim mesmo com o consolo raivoso e inútil de que um homem inteligente não pode, a sério, se tornar coisa alguma, e que só o tolo se torna alguma coisa.", 0.95, 0.5],
        ["leitor", "Sim, senhores: o homem inteligente do século dezenove precisa, e tem a obrigação moral, de ser uma criatura essencialmente sem caráter; e o homem de caráter, o homem de ação, uma criatura essencialmente limitada.", 0.95, 0.5],
        ["leitor", "Esta é a minha convicção de quarenta anos. Tenho agora quarenta anos, e quarenta anos são uma vida inteira; é a mais profunda velhice. Viver além dos quarenta é indecente, vulgar, imoral!", 0.95, 0.5],
        ["leitor", "Quem vive mais de quarenta anos? Respondam com sinceridade, com honestidade! Eu lhes digo quem vive: os tolos e os canalhas vivem.", 0.95, 0.5],
        ["leitor", "(Corte: ele repete, aos gritos, que dirá isso na cara de todos os velhos respeitáveis e do mundo inteiro.)", 1.0, 0.6],
        ["leitor", "Eu tenho o direito de falar assim, porque eu mesmo vou viver até os sessenta. Vou viver até os setenta! Vou viver até os oitenta!... Esperem! Deixem-me tomar fôlego...", 0.95, 1.0],
        ["leitor", "Os senhores provavelmente acham que eu quero fazê-los rir? Também nisso se enganaram. Não sou de jeito nenhum um homem tão alegre quanto lhes parece, ou quanto talvez lhes pareça;", 0.95, 0.5],
        ["leitor", "aliás, se os senhores, irritados com toda esta tagarelice (e eu já sinto que estão irritados), resolverem me perguntar: afinal, quem é você? Eu lhes respondo: sou um assessor de colégio.", 0.95, 0.5],
        ["leitor", "Trabalhei para ter o que comer (mas unicamente para isso), e quando, no ano passado, um parente distante me deixou seis mil rublos em testamento, pedi demissão na mesma hora e me instalei no meu canto.", 0.95, 0.5],
        ["leitor", "Eu já morava antes neste canto, mas agora me instalei neste canto. Meu quarto é ruim, horrível, na beira da cidade.", 0.95, 0.5],
        ["leitor", "(Corte: ele descreve a criada e os conselhos para deixar Petersburgo, por causa do clima e do custo de vida.)", 1.0, 0.6],
        ["leitor", "Mas eu fico em Petersburgo; não vou sair de Petersburgo! Não vou sair porque... Ah! Mas isso é absolutamente indiferente: se eu saio ou não saio.", 0.95, 1.0],
        ["leitor", "Pois bem: sobre o que um homem decente pode falar com mais prazer?", 0.95, 1.2],
        ["leitor", "Resposta: sobre si mesmo.", 0.95, 0.8],
        ["leitor", "Pois então eu também vou falar de mim.", 0.95, 1.5],
        ["comentarista", "Um fato para situar o narrador: assessor de colégio era o oitavo dos catorze graus da Tabela de Patentes, a escada do funcionalismo russo criada por Pedro, o Grande. Um posto do meio, sem brilho.", 1.0, 0.5],
        ["comentarista", "O funcionário pequeno é uma tradição da literatura russa. Está em O Capote, de Gógol, e no primeiro romance do próprio Dostoiévski, Gente Pobre, de 1846.", 1.0, 0.5],
        ["comentarista", "A diferença é que o homem do subsolo não pede pena a ninguém. Ele ataca, e ataca primeiro.", 1.0, 0.5],
        ["comentarista", "Um último detalhe: o narrador diz que viver além dos quarenta é indecente. Quando escreveu isso, Dostoiévski tinha quarenta e dois anos.", 1.0, 1.5],
    ]},
    2: {"titulo": "A consciência como doença", "trechos": [
        ["comentarista", "Memórias do Subsolo. Parte um, capítulo dois.", 1.0, 0.8],
        ["comentarista", "No capítulo anterior, o narrador disse que não conseguiu ser nada, nem mesmo um inseto. Agora ele vai explicar por quê.", 1.0, 1.0],
        ["leitor", "Capítulo dois.", 0.95, 0.8],
        ["leitor", "Agora quero lhes contar, senhores, queiram ou não queiram ouvir, por que eu nem mesmo um inseto consegui me tornar. Digo-lhes solenemente que muitas vezes quis me tornar um inseto. Mas nem disso fui digno.", 0.95, 0.5],
        ["leitor", "Juro-lhes, senhores, que ter consciência demais é uma doença; uma doença de verdade, completa.", 0.95, 0.5],
        ["leitor", "Para a vida humana comum, bastaria de sobra a consciência humana comum, isto é, a metade, um quarto da porção que cabe ao homem culto do nosso infeliz século dezenove,", 0.95, 0.5],
        ["leitor", "sobretudo se ele tem a dupla infelicidade de morar em Petersburgo, a cidade mais abstrata e mais premeditada de todo o globo terrestre. (Há cidades premeditadas e cidades não premeditadas.)", 0.95, 0.5],
        ["leitor", "Seria perfeitamente suficiente, por exemplo, a consciência com que vivem todas as chamadas pessoas espontâneas e os homens de ação.", 0.95, 0.5],
        ["leitor", "Aposto que os senhores pensam que escrevo tudo isso para me exibir, para fazer graça às custas dos homens de ação, e que, por exibicionismo de mau gosto, faço barulho com o sabre, como o meu oficial.", 0.95, 0.5],
        ["leitor", "Mas, senhores, quem pode se envaidecer das próprias doenças, e ainda se exibir com elas?", 0.95, 0.5],
        ["leitor", "Aliás, o que estou dizendo? Todos fazem isso; é justamente das doenças que as pessoas se envaidecem, e eu talvez mais que todos. Não vamos discutir; a minha objeção é absurda.", 0.95, 0.5],
        ["leitor", "Mas, mesmo assim, estou firmemente convencido de que não só consciência demais, mas qualquer consciência é uma doença. Insisto nisso.", 0.95, 1.2],
        ["comentarista", "Petersburgo foi fundada em 1703, por ordem de Pedro, o Grande, num pântano à beira do mar, para ser a capital moderna da Rússia. Daí a piada: é uma cidade feita de propósito, quase uma ideia de cidade.", 1.0, 0.5],
        ["comentarista", "Antes de Dostoiévski, Púchkin já tinha feito de Petersburgo uma personagem, no poema O Cavaleiro de Bronze, e Gógol, nos seus contos de Petersburgo.", 1.0, 1.2],
        ["leitor", "Deixemos isso por um minuto. Digam-me uma coisa: por que acontecia, como que de propósito, que naqueles mesmos minutos, sim, nos mesmíssimos minutos,", 0.95, 0.5],
        ["leitor", "em que eu era mais capaz de perceber todas as sutilezas de tudo o que é belo e sublime, como se dizia entre nós em outros tempos,", 0.95, 0.5],
        ["leitor", "acontecia de eu já não perceber, e sim fazer coisas tão feias, coisas que... bem, numa palavra, que talvez todos façam, mas que, como de propósito, me vinham justamente quando eu tinha mais consciência de que não devia fazê-las de modo algum?", 0.95, 0.5],
        ["leitor", "Quanto mais consciência eu tinha do bem e de todo esse belo e sublime, mais fundo eu afundava no meu lodo, e mais capaz ficava de me atolar nele por completo.", 0.95, 0.5],
        ["leitor", "Mas o traço principal era que tudo isso parecia não estar em mim por acaso, mas como se tivesse de ser assim. Como se fosse o meu estado mais normal, e de modo algum uma doença ou um defeito,", 0.95, 0.5],
        ["leitor", "de modo que, por fim, até perdi a vontade de lutar contra esse defeito. Acabei quase acreditando (e talvez tenha mesmo acreditado) que esse era, talvez, o meu estado normal.", 0.95, 0.5],
        ["leitor", "Mas antes, no começo, quanto tormento eu suportei nessa luta! Eu não acreditava que isso acontecesse com os outros, e por isso guardei a vida inteira como um segredo. Eu tinha vergonha (talvez ainda tenha);", 0.95, 0.5],
        ["leitor", "chegava ao ponto de sentir um prazerzinho secreto, anormal, mesquinho, ao voltar para o meu canto numa daquelas noites mais sórdidas de Petersburgo,", 0.95, 0.5],
        ["leitor", "e ter a consciência aguda de que hoje, de novo, eu tinha feito uma sujeira, e que o que estava feito não tinha mais volta;", 0.95, 0.5],
        ["leitor", "e, por dentro, em segredo, roer, roer a mim mesmo com os dentes, serrar e chupar a mim mesmo, até que a amargura se transformava, enfim, numa doçura vergonhosa, maldita, e, por fim, num prazer decidido, sério!", 0.95, 0.5],
        ["leitor", "Sim, num prazer, num prazer! Insisto nisso.", 0.95, 1.2],
        ["comentarista", "A expressão o belo e o sublime vem da estética do século dezoito, de Kant e de Schiller, e virou bordão dos idealistas russos dos anos 1840.", 1.0, 0.5],
        ["comentarista", "Dostoiévski foi um deles. Em 1849, foi preso por frequentar um círculo de discussão, o de Petrachévski, passou por uma falsa execução e cumpriu quatro anos de trabalhos forçados na Sibéria.", 1.0, 0.5],
        ["comentarista", "Isso é fato. Agora, uma interpretação: aqui ele parece rir do idealista que ele mesmo foi, o homem que sente o sublime e, no minuto seguinte, faz a coisa feia.", 1.0, 1.2],
        ["leitor", "Comecei a falar justamente porque quero saber ao certo: será que os outros também têm prazeres assim? Vou lhes explicar: o prazer vinha justamente da consciência viva demais da própria humilhação;", 0.95, 0.5],
        ["leitor", "de sentir que já se chegou à última parede; que isso é ruim, mas que não pode ser de outro jeito; que não há saída, que você nunca vai se tornar outra pessoa;", 0.95, 0.5],
        ["leitor", "que, mesmo que ainda houvesse tempo e fé para se transformar em outra coisa, você mesmo, com certeza, não ia querer se transformar;", 0.95, 0.5],
        ["leitor", "e, se quisesse, também não faria nada, porque, na verdade, talvez nem exista em que se transformar.", 0.95, 0.5],
        ["leitor", "(Corte: ele atribui tudo às leis da consciência intensificada, e à inércia que vem delas.)", 1.0, 0.6],
        ["leitor", "Mas chega... Ah, quanta coisa eu amontoei, e o que expliquei?... Como se explica aqui o prazer? Mas eu vou me explicar! Vou levar isso até o fim! Foi para isso mesmo que peguei a pena...", 0.95, 1.0],
        ["leitor", "Eu, por exemplo, sou terrivelmente vaidoso. Sou desconfiado e suscetível como um corcunda ou um anão, mas, palavra, tive momentos em que, se me dessem uma bofetada, talvez eu até ficasse contente.", 0.95, 0.5],
        ["leitor", "Falo sério: com certeza eu saberia encontrar até nisso o meu tipo de prazer; naturalmente, o prazer do desespero. Mas é no desespero que estão os prazeres mais ardentes,", 0.95, 0.5],
        ["leitor", "sobretudo quando se tem uma consciência muito forte de que a situação não tem saída. E ali, com a bofetada, a consciência de como fomos reduzidos a pó esmaga de vez.", 0.95, 0.5],
        ["leitor", "O principal, porém, é que, por mais voltas que se dê, sempre acabo sendo o primeiro culpado de tudo e, o que é mais ofensivo, culpado sem culpa e, por assim dizer, pelas leis da natureza.", 0.95, 0.5],
        ["leitor", "Culpado, em primeiro lugar, porque sou mais inteligente do que todos os que me cercam.", 0.95, 0.5],
        ["leitor", "(Eu sempre me considerei mais inteligente do que todos os que me cercam, e às vezes, acreditem, até me envergonhava disso. Pelo menos, a vida inteira olhei meio de lado, e nunca consegui olhar as pessoas nos olhos.)", 0.95, 0.5],
        ["leitor", "Culpado, enfim, porque, se houvesse em mim alguma generosidade, eu só teria mais tormento com a consciência de toda a sua inutilidade.", 0.95, 0.5],
        ["leitor", "Afinal, eu com certeza não saberia fazer nada com a minha generosidade: nem perdoar, porque o ofensor talvez tenha me batido pelas leis da natureza, e as leis da natureza não se perdoam;", 0.95, 0.5],
        ["leitor", "nem esquecer, porque, ainda que sejam as leis da natureza, ofende do mesmo jeito.", 0.95, 0.5],
        ["leitor", "Por fim, mesmo que eu quisesse ser nada generoso e, ao contrário, desejasse me vingar do ofensor, eu não conseguiria me vingar de ninguém em nada, porque com certeza não me decidiria a fazer coisa alguma, mesmo que pudesse.", 0.95, 0.5],
        ["leitor", "Por que não me decidiria? Sobre isso, quero dizer duas palavras à parte.", 0.95, 1.5],
        ["comentarista", "Guarde duas imagens deste capítulo, porque elas voltam: a última parede, onde ele diz ter chegado, e as leis da natureza, que não dá para perdoar. No capítulo três, a parede vira o centro da discussão.", 1.0, 0.5],
        ["comentarista", "Fora do livro, dois ecos. Dois anos depois, em 1866, Dostoiévski publicou Crime e Castigo, cujo herói, Raskólnikov, também pensa demais antes de agir.", 1.0, 0.5],
        ["comentarista", "E em 1887 Nietzsche encontrou este livro, por acaso, numa tradução francesa, e contou numa carta ao amigo Overbeck que reconheceu ali um parente. É um fato curioso; não prova que um influenciou o outro.", 1.0, 1.5],
    ]},
}

for n, c in CAPITULOS.items():
    m = {v: sum(len(t[1].split()) for t in c["trechos"] if t[0] == v) / 140 for v in VOZES}
    print(f"Capítulo {n}, {c['titulo']}: texto {m['leitor']:.1f} min, comentários {m['comentarista']:.1f} min.")

In [ ]:
# ── 6. Criar as duas vozes e ouvir (troque a semente se não gostar) ─────────
import hashlib, os, requests
from IPython.display import Audio, display

def gerar(texto, destino, **campos):
    arquivos = campos.pop("files", None)
    dados = {"text": texto, "language": "pt", "effect_preset": "raw", **campos}
    r = requests.post(f"{BASE}/generate", data=dados, files=arquivos, timeout=1800)
    if r.status_code != 200:
        raise SystemExit(f"Erro {r.status_code} em /generate: {r.text[:500]}\nLog: {LOG_PATH}")
    with open(destino, "wb") as f:
        f.write(r.content)

def chave(*partes):
    return hashlib.sha1(repr(partes).encode()).hexdigest()[:8]

REF = {}   # voz -> (wav de referência, texto)
for nome, v in VOZES.items():
    texto = next(max(falas, key=len) for _, c in sorted(CAPITULOS.items())
                 if (falas := [t[1] for t in c["trechos"] if t[0] == nome]))
    wav = os.path.join(PASTA_VOZES, f"{nome}_{chave(v, texto)}.wav")
    if not os.path.exists(wav):
        gerar(texto, wav, instruct=v["voz"], seed=v["semente"])
    REF[nome] = (wav, texto)
    print(f"{nome} ({v['voz']}, semente {v['semente']}): {texto}")
    display(Audio(wav))

In [ ]:
# ── 7. Narrar e montar cada capítulo (guiada e limpa) ───────────────────────
import os, subprocess
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def narrar(n):
    pasta = os.path.join(LIVRO, f"cap{n:02d}", "trechos")
    os.makedirs(pasta, exist_ok=True)
    saida = []
    for i, (quem, texto, vel, pausa) in enumerate(CAPITULOS[n]["trechos"], 1):
        ref_wav, ref_texto = REF[quem]
        wav = os.path.join(pasta, f"{i:03d}_{quem}_{chave(os.path.basename(ref_wav), texto, vel)}.wav")
        if not os.path.exists(wav):
            with open(ref_wav, "rb") as ref:
                gerar(texto, wav, seed=VOZES[quem]["semente"], ref_text=ref_texto, speed=vel,
                      files={"ref_audio": ("ref.wav", ref, "audio/wav")})
            print(f"{n}.{i:03d} {quem}: {texto}")
        saida.append((quem, wav, pausa))
    return saida

def _nota(f, dur, vol, sr):
    t = np.arange(int(dur * sr)) / sr
    env = np.clip(t / 0.02, 0, 1) * np.exp(-4 * t)
    return (vol * (np.sin(2 * np.pi * f * t) + 0.2 * np.sin(4 * np.pi * f * t)) * env).astype("float32")

def sinal(sr, entra):
    # Duas notas curtas e baixas: sobem quando o comentário começa, descem quando acaba.
    fs = (440.0, 659.25) if entra else (659.25, 440.0)
    out = np.zeros(int(1.0 * sr), dtype="float32")
    for k, f in enumerate(fs):
        nota = _nota(f, 0.6, 0.06, sr)
        i = int(k * 0.18 * sr)
        out[i:i + len(nota)] += nota[:len(out) - i]
    return out

def exportar(audio, sr, pasta, nome):
    tmp = os.path.join(pasta, nome + ".wav")
    sf.write(tmp, audio, sr)
    mp3 = os.path.join(pasta, nome + ".mp3")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "44100",
                    "-codec:a", "libmp3lame", "-q:a", "2", mp3], check=True)
    os.remove(tmp)
    print(f"{nome}.mp3: {len(audio) / sr / 60:.1f} min")
    display(Audio(mp3))

def montar(n, trechos):
    sr = None
    guiada, limpa, anterior = [], [], "leitor"
    for quem, wav, pausa in trechos:
        a, s = sf.read(wav, dtype="float32", always_2d=True)
        sr = sr or s
        assert s == sr, f"taxa de amostragem diferente em {wav}"
        fala = a.mean(axis=1)
        silencio = np.zeros(int(pausa * sr), dtype="float32")
        if quem != anterior:
            guiada += [sinal(sr, entra=(quem == "comentarista")), np.zeros(int(0.3 * sr), dtype="float32")]
        guiada += [fala, silencio]
        if quem == "leitor":
            limpa += [fala, silencio]
        anterior = quem
    if anterior == "comentarista":
        guiada.append(sinal(sr, entra=False))
    um_seg = np.zeros(sr, dtype="float32")
    pasta = os.path.join(LIVRO, f"cap{n:02d}")
    exportar(np.concatenate([um_seg, *guiada, um_seg]), sr, pasta, f"cap{n:02d}_guiada")
    exportar(np.concatenate([um_seg, *limpa, um_seg]), sr, pasta, f"cap{n:02d}_limpa")

REFAZER = False   # True: narra de novo capítulos que já têm os dois MP3

for n in CAPITULOS_PARA_GERAR:
    print(f"\n=== Capítulo {n}: {CAPITULOS[n]['titulo']} ===")
    prontos = [os.path.join(LIVRO, f"cap{n:02d}", f"cap{n:02d}_{v}.mp3") for v in ("guiada", "limpa")]
    if not REFAZER and all(map(os.path.exists, prontos)):
        print("Já estava pronto (para refazer, use REFAZER = True).")
        continue
    montar(n, narrar(n))
print("\nPronto! Os arquivos estão no Drive, em", LIVRO)

# Se um trecho ficou ruim, apague o .wav dele na pasta capNN/trechos/ e rode esta célula de novo.